# 3. ACC-Classwise Input-Normalized STR Opposite-Span Sensor Health Score Calculation

This notebook uses ACC responses as input-condition correction channels rather than as direct health-score targets. The central idea is to reduce residual score variability within the STR comparison groups by normalizing each STR spectral response vector with the local ACC frequency-class input measured at the same span and side.


## Method Overview

The method starts from the sensor-time class-ratio-weighted response vector, `r_tilde`. Instead of directly comparing STR responses, the ACC response measured at the same `span + side` is used as the frequency-class input for the same event.

For each event, span, side, and frequency class, the ACC input amplitude is defined as:

```text
A_s,h,c(t) = sqrt( mean_{k in ACC(s,h)} r_tilde_ACC,k,c(t)^2 )
```

Here, `s` denotes span, `h` denotes side, and `c` denotes frequency class. The STR response at the same span and side is then divided by this class-wise ACC input.

```text
r_hat_STR,i,c(t) = r_tilde_STR,i,c(t) / A_s,h,c(t)
```

Group averaging is then performed only for the STR comparison groups.

```text
same_group_S_avg = mean S(i,j) within the same STR side, excluding self-pairs
opposite_span_S_avg = mean S(i,j) for sensors with the same STR side and opposite span
health = 100 x sqrt(same_group_S_avg x opposite_span_S_avg)
```

This formulation does not use manual thresholds, outlier rejection, or manually defined score ranges. The calculation is based on a single physical normalization rule: `class-wise STR response / class-wise local ACC input`. This allows STR responses to be compared relative to the measured excitation condition rather than by absolute response magnitude alone.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from IPython.display import display

WORKDIR = Path.cwd().resolve()
ROOT = next(
    (
        candidate
        for candidate in [WORKDIR, *WORKDIR.parents]
        if (candidate / 'AutoResearch_generated_method' / 'tables_v2').exists()
    ),
    WORKDIR,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.aquinas_common import method_sensor_index


def _apply_method_sensor_index(table):
    table = table.copy()
    if 'sensor_alias' not in table.columns:
        return table
    mapped_index = table['sensor_alias'].map(method_sensor_index)
    if mapped_index.notna().all():
        table['sensor_index'] = mapped_index.astype(int)
    elif 'sensor_index' in table.columns:
        valid = mapped_index.notna()
        table.loc[valid, 'sensor_index'] = mapped_index.loc[valid].astype(int)
    return table
TABLE_DIR = ROOT / 'AutoResearch_generated_method' / 'tables_v2'
FIG_DIR = ROOT / 'AutoResearch_generated_method' / 'figures_v2'
FIG_DIR.mkdir(parents=True, exist_ok=True)
CLASS_AWARE_HEALTH_PATH = TABLE_DIR / 'sensor_time_class_ratio_weighted_opposite_span_sensor_coherence_health_v1.csv'
ACC_INPUT_NORMALIZED_STR_HEALTH_PATH = TABLE_DIR / 'acc_classwise_input_normalized_str_opposite_span_sensor_coherence_health_v1.csv'
ACC_CLASS_INPUT_MAP_PATH = TABLE_DIR / 'acc_span_side_class_input_rms_map_v1.csv'

TARGET_EVENT_PATH = (
    ROOT
    / 'EWSHM_dataset_preprocessed_event_level'
    / 'AQUINAS_SET2_2023_04'
    / 'OLD'
    / 'PART020'
    / 'AQUINAS_SET2_OLD_2023_04_PART020_EVENT028_ID001266.hdf5'
)

CLASS_NUMBERS = [1, 2, 3, 4, 5]
CLASS_LABELS = [f'Class {idx}' for idx in CLASS_NUMBERS]
CLASS_RESPONSE_COLS = [f'class_{idx}_energy_duration_amplitude' for idx in CLASS_NUMBERS]
SENSOR_CLASS_WEIGHT_COLS = [f'sensor_class_{idx}_weight' for idx in CLASS_NUMBERS]
ACC_CLASS_INPUT_COLS = [f'acc_class_{idx}_input_rms' for idx in CLASS_NUMBERS]
PLOT_TITLE_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
DECK_COLORS = {'NEW': '#1f77b4', 'OLD': '#d62728'}
EPS = 1e-30

if not CLASS_AWARE_HEALTH_PATH.exists():
    raise FileNotFoundError(f'Missing required class-aware health table: {CLASS_AWARE_HEALTH_PATH}')


def _needs_rebuild(output_path, input_paths, force=False):
    output_path = Path(output_path)
    if force or not output_path.exists():
        return True
    output_mtime = output_path.stat().st_mtime
    return any(Path(input_path).stat().st_mtime > output_mtime for input_path in input_paths)


def _sensor_similarity_matrices(response_map):
    response_map = np.asarray(response_map, dtype=float)
    norms = np.linalg.norm(response_map, axis=1)
    valid = np.isfinite(norms) & (norms > 0)
    unit_map = np.zeros_like(response_map, dtype=float)
    unit_map[valid] = response_map[valid] / norms[valid, None]

    cosine_similarity = np.clip(unit_map @ unit_map.T, 0.0, 1.0)
    cosine_similarity[~(valid[:, None] & valid[None, :])] = np.nan

    denom = norms[:, None] + norms[None, :]
    magnitude_similarity = np.zeros_like(cosine_similarity, dtype=float)
    pair_valid = valid[:, None] & valid[None, :] & (denom > 0)
    magnitude_similarity[pair_valid] = (
        2.0
        * np.minimum(norms[:, None], norms[None, :])[pair_valid]
        / (denom[pair_valid] + EPS)
    )
    magnitude_similarity = np.clip(magnitude_similarity, 0.0, 1.0)
    magnitude_similarity[~pair_valid] = np.nan

    sensor_similarity = np.clip(cosine_similarity * magnitude_similarity, 0.0, 1.0)
    sensor_similarity[~pair_valid] = np.nan
    return cosine_similarity, magnitude_similarity, sensor_similarity, norms


def _mean_at_indices(matrix, row_idx, col_indices):
    if len(col_indices) == 0:
        return np.nan
    values = np.asarray(matrix[row_idx, col_indices], dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    return float(np.mean(values))


def _safe_divide_matrix(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    safe_denominator = np.where(np.isfinite(denominator) & (denominator > 0), denominator, np.nan)
    return numerator / safe_denominator


def _format_colorbar_endpoint(value):
    value = float(value)
    if not np.isfinite(value):
        return 'nan'
    abs_value = abs(value)
    if abs_value < 1e-12:
        return '0'
    if abs_value >= 100:
        return f'{value:.0f}'
    if abs_value >= 10:
        return f'{value:.1f}'
    if abs_value >= 1:
        return f'{value:.2f}'
    if abs_value >= 0.01:
        return f'{value:.3f}'
    return f'{value:.4g}'


def _set_colorbar_min_max_ticks(cbar, im=None, vmin=None, vmax=None):
    if vmin is None or vmax is None:
        if im is None:
            return cbar
        vmin, vmax = im.get_clim()
    endpoints = [float(vmin), float(vmax)]
    if np.isclose(endpoints[0], endpoints[1]):
        cbar.set_ticks([endpoints[0]])
        cbar.set_ticklabels([_format_colorbar_endpoint(endpoints[0])])
    else:
        cbar.set_ticks(endpoints)
        cbar.set_ticklabels([_format_colorbar_endpoint(value) for value in endpoints])
    return cbar


def _add_bottom_colorbar(fig, cax, im, label):
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal')
    cbar.set_label(label, fontsize=9)
    _set_colorbar_min_max_ticks(cbar, im=im)
    cbar.ax.tick_params(labelsize=8, length=2.2, pad=1.2)
    return cbar


def _format_class_axis(ax, row_labels, xlabel='Frequency class'):
    ax.set_xticks(np.arange(len(CLASS_LABELS)))
    ax.set_xticklabels(CLASS_LABELS, fontsize=8.6, rotation=0, ha='center')
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.set_ylabel('Method-level sensor index', fontsize=11)
    ax.tick_params(length=1.8, pad=3.0)


def _format_similarity_axis(ax, row_labels):
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks(np.arange(len(row_labels)))
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_xticklabels(row_labels, fontsize=10.0, rotation=90)
    ax.set_yticklabels(row_labels, fontsize=10.0)
    ax.tick_params(length=1.8, pad=1.6)


def _finite_vmax(values, percentile=99, fallback=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) == 0:
        return fallback
    return max(1e-12, float(np.nanpercentile(values, percentile)))


## 3.1 Build ACC-Classwise Input-Normalized STR Health Table

This section reconstructs the class-weighted response table, estimates the ACC class RMS input map for the same event, span, and side, and normalizes the STR response by the corresponding class-wise ACC input. The similarity matrix `S(i,j)` and the health score are then recalculated only for the G1/G2 STR sensor groups.


In [ ]:
EVENT_KEYS = ['path', 'deck', 'set_name', 'event_id', 'campaign_month', 'part_index', 'event_sequence']
SOURCE_USECOLS = [
    *EVENT_KEYS,
    'deck_event_count',
    'sensor_index',
    'sensor_alias',
    'support_group',
    'quantity',
    'span',
    'side',
    'location',
    'axis',
    'local_group_id',
    'comparison_group',
    'dominant_class',
    'dominant_band_signal_rms',
    'row_psd_energy',
    'active_duration_s',
    'sensor_time_dominant_class',
    'sensor_time_class_confidence',
    'sensor_time_class_entropy',
    'same_group_S_avg',
    'opposite_span_S_avg',
    'sensor_health_score',
    *SENSOR_CLASS_WEIGHT_COLS,
    *CLASS_RESPONSE_COLS,
]

source_preview = pd.read_csv(CLASS_AWARE_HEALTH_PATH, usecols=SOURCE_USECOLS, nrows=2000)
source_preview = _apply_method_sensor_index(source_preview)
str_preview = source_preview[source_preview['quantity'].eq('STR')].drop_duplicates('sensor_alias').copy()

LOCATION_ORDER = {'INF': 0, 'SHE': 1, 'SUP': 2, 'INT': 3, 'MID': 4}
SPAN_ORDER = {'S1': 0, 'S2': 1}
GROUP_ORDER = {'STR DO': 0, 'STR UP': 1}

str_preview['group_rank'] = str_preview['comparison_group'].map(GROUP_ORDER).astype(int)
str_preview['span_rank'] = str_preview['span'].map(SPAN_ORDER).fillna(99).astype(int)
str_preview['location_rank'] = str_preview['location'].map(LOCATION_ORDER).fillna(99).astype(int)
str_preview = str_preview.sort_values(['group_rank', 'span_rank', 'location_rank', 'sensor_index']).reset_index(drop=True)

str_sensor_alias_order = str_preview['sensor_alias'].tolist()
str_sensor_index_labels = str_preview['sensor_index'].astype(int).tolist()
str_sensor_meta = str_preview[[
    'sensor_index', 'sensor_alias', 'quantity', 'span', 'side', 'location', 'axis', 'local_group_id', 'comparison_group'
]].copy()
str_meta_ordered = str_sensor_meta.set_index('sensor_alias').reindex(str_sensor_alias_order).reset_index()

same_group_indices_by_sensor = []
opposite_span_indices_by_sensor = []
for idx, row in str_meta_ordered.iterrows():
    same_group_mask = str_meta_ordered['comparison_group'].eq(row['comparison_group']) & str_meta_ordered.index.to_series().ne(idx)
    opposite_span_mask = str_meta_ordered['comparison_group'].eq(row['comparison_group']) & str_meta_ordered['span'].ne(row['span'])
    same_group_indices_by_sensor.append(str_meta_ordered.index[same_group_mask].to_numpy(int))
    opposite_span_indices_by_sensor.append(str_meta_ordered.index[opposite_span_mask].to_numpy(int))

peer_count_table = str_meta_ordered[['sensor_index', 'sensor_alias', 'local_group_id', 'comparison_group', 'span', 'side', 'location']].copy()
peer_count_table['same_group_neighbor_count'] = [len(x) for x in same_group_indices_by_sensor]
peer_count_table['opposite_span_neighbor_count'] = [len(x) for x in opposite_span_indices_by_sensor]
print('Peer count summary by STR comparison group')
display(peer_count_table.groupby(['local_group_id', 'comparison_group'])[[
    'same_group_neighbor_count', 'opposite_span_neighbor_count'
]].agg(['min', 'max']))


def _build_acc_class_input_maps(source):
    response = source[CLASS_RESPONSE_COLS].to_numpy(float)
    weights = np.clip(source[SENSOR_CLASS_WEIGHT_COLS].to_numpy(float), 0.0, None)
    weighted_response = response * np.sqrt(weights)
    response_cols = [f'weighted_response_class_{idx}' for idx in CLASS_NUMBERS]
    for col_idx, col in enumerate(response_cols):
        source[col] = weighted_response[:, col_idx]

    acc_rows = source[source['quantity'].eq('ACC')].copy()
    if acc_rows.empty:
        raise RuntimeError('No ACC rows were found in the source table.')

    sq_cols = []
    for class_idx, response_col in zip(CLASS_NUMBERS, response_cols):
        sq_col = f'acc_class_{class_idx}_weighted_response_sq'
        acc_rows[sq_col] = acc_rows[response_col] ** 2
        sq_cols.append(sq_col)

    acc_span_side = acc_rows.groupby([*EVENT_KEYS, 'span', 'side'], as_index=False)[sq_cols].mean()
    acc_side = acc_rows.groupby([*EVENT_KEYS, 'side'], as_index=False)[sq_cols].mean()
    acc_event = acc_rows.groupby(EVENT_KEYS, as_index=False)[sq_cols].mean()

    rename_span_side = {}
    rename_side = {}
    rename_event = {}
    for class_idx, sq_col in zip(CLASS_NUMBERS, sq_cols):
        input_col = f'acc_class_{class_idx}_input_rms'
        side_col = f'{input_col}_side_fallback'
        event_col = f'{input_col}_event_fallback'
        acc_span_side[input_col] = np.sqrt(np.clip(acc_span_side[sq_col], 0.0, None))
        acc_side[side_col] = np.sqrt(np.clip(acc_side[sq_col], 0.0, None))
        acc_event[event_col] = np.sqrt(np.clip(acc_event[sq_col], 0.0, None))
        rename_span_side[sq_col] = input_col
        rename_side[sq_col] = side_col
        rename_event[sq_col] = event_col

    acc_span_side = acc_span_side[[*EVENT_KEYS, 'span', 'side', *ACC_CLASS_INPUT_COLS]]
    acc_side = acc_side[[*EVENT_KEYS, 'side', *[f'{col}_side_fallback' for col in ACC_CLASS_INPUT_COLS]]]
    acc_event = acc_event[[*EVENT_KEYS, *[f'{col}_event_fallback' for col in ACC_CLASS_INPUT_COLS]]]
    acc_span_side.to_csv(ACC_CLASS_INPUT_MAP_PATH, index=False)
    return source, acc_span_side, acc_side, acc_event


def build_or_load_acc_input_normalized_str_health_table(force=False):
    if not _needs_rebuild(ACC_INPUT_NORMALIZED_STR_HEALTH_PATH, [CLASS_AWARE_HEALTH_PATH], force=force):
        health = pd.read_csv(ACC_INPUT_NORMALIZED_STR_HEALTH_PATH, low_memory=False)
        health = _apply_method_sensor_index(health)
        acc_map = pd.read_csv(ACC_CLASS_INPUT_MAP_PATH, low_memory=False) if ACC_CLASS_INPUT_MAP_PATH.exists() else None
        return health, acc_map

    source = pd.read_csv(CLASS_AWARE_HEALTH_PATH, usecols=SOURCE_USECOLS, low_memory=False)
    source = _apply_method_sensor_index(source)
    source = source.sort_values(['deck', 'event_sequence', 'sensor_index']).reset_index(drop=True)
    source, acc_span_side, acc_side, acc_event = _build_acc_class_input_maps(source)

    str_rows = source[source['quantity'].eq('STR')].copy()
    str_rows = str_rows.merge(acc_span_side, on=[*EVENT_KEYS, 'span', 'side'], how='left')
    str_rows = str_rows.merge(acc_side, on=[*EVENT_KEYS, 'side'], how='left')
    str_rows = str_rows.merge(acc_event, on=EVENT_KEYS, how='left')

    for input_col in ACC_CLASS_INPUT_COLS:
        side_col = f'{input_col}_side_fallback'
        event_col = f'{input_col}_event_fallback'
        invalid = ~np.isfinite(str_rows[input_col]) | (str_rows[input_col] <= 0)
        str_rows.loc[invalid, input_col] = str_rows.loc[invalid, side_col]
        invalid = ~np.isfinite(str_rows[input_col]) | (str_rows[input_col] <= 0)
        str_rows.loc[invalid, input_col] = str_rows.loc[invalid, event_col]
        invalid = ~np.isfinite(str_rows[input_col]) | (str_rows[input_col] <= 0)
        if invalid.any():
            fallback = float(np.nanmedian(acc_span_side[input_col]))
            str_rows.loc[invalid, input_col] = fallback

    output_rows = []
    for _, event_df in str_rows.groupby('path', sort=False):
        ordered = event_df.set_index('sensor_alias').reindex(str_sensor_alias_order)
        if ordered[CLASS_RESPONSE_COLS].isna().all(axis=None):
            continue

        raw_response_map = ordered[CLASS_RESPONSE_COLS].to_numpy(float)
        sensor_class_weight_map = np.clip(ordered[SENSOR_CLASS_WEIGHT_COLS].to_numpy(float), 0.0, None)
        weighted_response_map = raw_response_map * np.sqrt(sensor_class_weight_map)
        acc_class_input_map = ordered[ACC_CLASS_INPUT_COLS].to_numpy(float)
        acc_normalized_response_map = _safe_divide_matrix(weighted_response_map, acc_class_input_map)

        cosine_similarity, magnitude_similarity, sensor_similarity, _ = _sensor_similarity_matrices(acc_normalized_response_map)

        for sensor_idx, (sensor_alias, row) in enumerate(ordered.iterrows()):
            same_idx = same_group_indices_by_sensor[sensor_idx]
            opposite_idx = opposite_span_indices_by_sensor[sensor_idx]
            same_group_S = _mean_at_indices(sensor_similarity, sensor_idx, same_idx)
            opposite_span_S = _mean_at_indices(sensor_similarity, sensor_idx, opposite_idx)
            if np.isfinite(same_group_S) and np.isfinite(opposite_span_S):
                score = float(100.0 * np.sqrt(max(same_group_S, 0.0) * max(opposite_span_S, 0.0)))
            else:
                available = np.asarray([same_group_S, opposite_span_S], dtype=float)
                available = available[np.isfinite(available)]
                score = float(100.0 * np.nanmean(available)) if len(available) else np.nan

            without_acc_score = float(row['sensor_health_score'])
            output_rows.append({
                **{key: row[key] for key in EVENT_KEYS},
                'deck_event_count': int(row['deck_event_count']),
                'sensor_index': int(row['sensor_index']),
                'sensor_alias': sensor_alias,
                'support_group': row['support_group'],
                'quantity': row['quantity'],
                'span': row['span'],
                'side': row['side'],
                'location': row['location'],
                'axis': row['axis'],
                'local_group_id': row['local_group_id'],
                'comparison_group': row['comparison_group'],
                'dominant_class': int(row['dominant_class']),
                'dominant_band_signal_rms': float(row['dominant_band_signal_rms']),
                'row_psd_energy': float(row['row_psd_energy']),
                'active_duration_s': float(row['active_duration_s']),
                'sensor_time_dominant_class': int(row['sensor_time_dominant_class']),
                'sensor_time_class_confidence': float(row['sensor_time_class_confidence']),
                'sensor_time_class_entropy': float(row['sensor_time_class_entropy']),
                'same_group_neighbor_count': int(len(same_idx)),
                'opposite_span_neighbor_count': int(len(opposite_idx)),
                'same_group_pattern_similarity': _mean_at_indices(cosine_similarity, sensor_idx, same_idx),
                'same_group_magnitude_similarity': _mean_at_indices(magnitude_similarity, sensor_idx, same_idx),
                'same_group_S_avg': same_group_S,
                'opposite_span_pattern_similarity': _mean_at_indices(cosine_similarity, sensor_idx, opposite_idx),
                'opposite_span_magnitude_similarity': _mean_at_indices(magnitude_similarity, sensor_idx, opposite_idx),
                'opposite_span_S_avg': opposite_span_S,
                'without_acc_sensor_health_score': without_acc_score,
                'sensor_health_score': score,
                'score_delta_from_without_acc': score - without_acc_score if np.isfinite(score) and np.isfinite(without_acc_score) else np.nan,
                **{col: float(row[col]) for col in ACC_CLASS_INPUT_COLS},
                **{col: float(row[col]) for col in SENSOR_CLASS_WEIGHT_COLS},
                **{col: float(row[col]) for col in CLASS_RESPONSE_COLS},
            })

    health = pd.DataFrame(output_rows)
    health.to_csv(ACC_INPUT_NORMALIZED_STR_HEALTH_PATH, index=False)
    return health, acc_span_side


health_sensors, acc_class_input_map = build_or_load_acc_input_normalized_str_health_table(force=False)
print(f'ACC-classwise input-normalized STR health table: {ACC_INPUT_NORMALIZED_STR_HEALTH_PATH}')
print(f'ACC class input RMS map: {ACC_CLASS_INPUT_MAP_PATH}')
print(f'Rows: {len(health_sensors):,}')
print(f'Events: {health_sensors["path"].nunique():,}')
print(f'Sensors: {health_sensors["sensor_alias"].nunique()}')
print('Score formula: 100 * sqrt(same_group_S_avg * opposite_span_S_avg), using class-wise STR / local ACC input response.')

score_summary = health_sensors.groupby(['local_group_id', 'comparison_group', 'deck']).agg({
    'sensor_health_score': ['count', 'mean', 'median', 'std'],
    'without_acc_sensor_health_score': ['mean', 'median', 'std'],
    'score_delta_from_without_acc': ['mean', 'median', 'std'],
})
display(score_summary.round(6))

if acc_class_input_map is not None:
    display(acc_class_input_map.groupby(['span', 'side', 'deck'])[ACC_CLASS_INPUT_COLS].median().round(6))


## 3.2 STR and ACC Spectral Response Vector Maps

The first figure shows the raw STR and ACC spectral response vectors before applying sensor-specific class weights. The second figure shows the same event after multiplying each spectral class by the square root of its sensor-specific class weight.


In [ ]:

def _read_target_event_source_rows(path):
    path = str(path)
    chunks = []
    for chunk in pd.read_csv(CLASS_AWARE_HEALTH_PATH, usecols=SOURCE_USECOLS, chunksize=250_000, low_memory=False):
        matched = chunk[chunk['path'].eq(path)].copy()
        if not matched.empty:
            chunks.append(matched)
    if not chunks:
        raise RuntimeError(f'Target event was not found in the class-aware health source table: {path}')
    return pd.concat(chunks, ignore_index=True)


def _ordered_response_rows(table, alias_order=None):
    table = table.copy()
    if alias_order is None:
        table = table.sort_values('sensor_index').reset_index(drop=True)
    else:
        ordered_aliases = [alias for alias in alias_order if alias in set(table['sensor_alias'])]
        missing_aliases = sorted(set(table['sensor_alias']) - set(ordered_aliases))
        if missing_aliases:
            raise RuntimeError(f'Missing aliases in response order: {missing_aliases}')
        table = table.set_index('sensor_alias').reindex(ordered_aliases).reset_index()

    raw_response = table[CLASS_RESPONSE_COLS].to_numpy(float)
    class_weight = np.clip(table[SENSOR_CLASS_WEIGHT_COLS].to_numpy(float), 0.0, None)
    weighted_response = raw_response * np.sqrt(class_weight)
    return table, raw_response, weighted_response


def _weighted_response_rows(table, alias_order=None):
    table, _, weighted_response = _ordered_response_rows(table, alias_order=alias_order)
    return table, weighted_response


def _acc_grouped_alias_order(table):
    table = table.copy()
    acc_group_order = {'ACC DO': 0, 'ACC UP': 1}
    axis_order = {'Y': 0, 'Z': 1}
    table['group_rank'] = table['comparison_group'].map(acc_group_order).fillna(99).astype(int)
    table['span_rank'] = table['span'].map(SPAN_ORDER).fillna(99).astype(int)
    table['location_rank'] = table['location'].map(LOCATION_ORDER).fillna(99).astype(int)
    table['axis_rank'] = table['axis'].map(axis_order).fillna(99).astype(int)
    table = table.sort_values([
        'group_rank',
        'span_rank',
        'location_rank',
        'axis_rank',
        'sensor_index',
    ]).reset_index(drop=True)
    return table['sensor_alias'].tolist()


def _format_response_map_axis(ax, table, show_xlabel=False):
    ax.set_yticks(np.arange(len(table)))
    ax.set_yticklabels(table['sensor_index'].astype(int).tolist(), fontsize=11)
    ax.set_ylabel('Sensor index', fontsize=13)
    ax.set_xticks(np.arange(len(CLASS_LABELS)))
    ax.set_xticklabels(CLASS_LABELS if show_xlabel else [], fontsize=11)
    ax.set_xticks(np.arange(-0.5, len(CLASS_LABELS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(table), 1), minor=True)
    ax.grid(which='minor', color='0.70', linewidth=0.45, alpha=0.65)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.tick_params(axis='both', length=2.8, pad=2.0)
    if show_xlabel:
        ax.set_xlabel('Spectral response class', fontsize=13)


# Backward-compatible alias for existing notebook text/snippets.
_format_weighted_response_axis = _format_response_map_axis


def _plot_str_acc_response_maps(title, figure_name, cbar_label, scale_limits, str_matrix, acc_matrix):
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.8), constrained_layout=False, sharex=True)
    fig.subplots_adjust(left=0.095, right=0.985, bottom=0.125, top=0.900, hspace=0.22)
    fig.suptitle(title, fontsize=PLOT_TITLE_FONTSIZE, y=0.970)

    cmap_response = plt.get_cmap('jet').copy()
    cmap_response.set_bad('#f2f2f2')
    plot_specs = [
        (axes[0], str_matrix, str_weight_table, 'STR'),
        (axes[1], acc_matrix, acc_weight_table, 'ACC'),
    ]
    for ax, matrix, table, quantity in plot_specs:
        vmin, vmax = scale_limits[quantity]
        im = ax.imshow(
            np.ma.masked_invalid(matrix),
            aspect='auto',
            interpolation='nearest',
            cmap=cmap_response,
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_title(quantity, fontsize=15, pad=6)
        _format_response_map_axis(ax, table, show_xlabel=(quantity == 'ACC'))
        cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.024, pad=0.015)
        cbar.set_label(cbar_label, fontsize=8.5)
        _set_colorbar_min_max_ticks(cbar, vmin=vmin, vmax=vmax)
        cbar.ax.tick_params(labelsize=9, length=2.2, pad=1.2)

    fig_path = FIG_DIR / figure_name
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig_path


RAW_RESPONSE_SCALE_LIMITS = {
    'STR': (0.0, 0.0130),
    'ACC': (0.0, 0.0025),
}
WEIGHTED_RESPONSE_SCALE_LIMITS = {
    'STR': (0.0, 0.0110),
    'ACC': (0.0, 0.0019),
}

target_event_source = _read_target_event_source_rows(TARGET_EVENT_PATH)
target_event_source = _apply_method_sensor_index(target_event_source)
target_event_source['quantity'] = target_event_source['quantity'].astype(str)
str_target_rows = target_event_source[target_event_source['quantity'].eq('STR')]
acc_target_rows = target_event_source[target_event_source['quantity'].eq('ACC')]

acc_weight_alias_order = _acc_grouped_alias_order(acc_target_rows)
str_weight_table, str_raw_response_map, str_weighted_response_map = _ordered_response_rows(
    str_target_rows,
    alias_order=str_sensor_alias_order,
)
acc_weight_table, acc_raw_response_map, acc_weighted_response_map = _ordered_response_rows(
    acc_target_rows,
    alias_order=acc_weight_alias_order,
)

raw_response_fig_path = _plot_str_acc_response_maps(
    '3.2a STR and ACC Spectral Response Vector Map',
    '3_2a_str_acc_spectral_response_vector_map.png',
    'Spectral response',
    RAW_RESPONSE_SCALE_LIMITS,
    str_raw_response_map,
    acc_raw_response_map,
)

print('Selected event for raw STR/ACC response map')
print(f'  path = {TARGET_EVENT_PATH.relative_to(ROOT)}')
print(f'  STR raw response map shape = {str_raw_response_map.shape[0]} sensors x {str_raw_response_map.shape[1]} classes')
print(f"  STR row order = {str_weight_table['sensor_index'].astype(int).tolist()}")
print(f'  STR color scale = {RAW_RESPONSE_SCALE_LIMITS["STR"][0]:.4f} to {RAW_RESPONSE_SCALE_LIMITS["STR"][1]:.4f}')
print(f'  ACC raw response map shape = {acc_raw_response_map.shape[0]} sensors x {acc_raw_response_map.shape[1]} classes')
print(f"  ACC row order = {acc_weight_table['sensor_index'].astype(int).tolist()}")
print(f'  ACC color scale = {RAW_RESPONSE_SCALE_LIMITS["ACC"][0]:.4f} to {RAW_RESPONSE_SCALE_LIMITS["ACC"][1]:.4f}')
print(f'  saved figure = {raw_response_fig_path}')


### 3.2b Class-Weighted STR and ACC Spectral Response Vector Map

This figure shows the same STR and ACC response vectors after applying the sensor-specific class weights.


In [ ]:

weighted_response_fig_path = _plot_str_acc_response_maps(
    '3.2b Class-Weighted STR and ACC Spectral Response Vector Map',
    '3_2b_class_weighted_str_acc_response_vector_map.png',
    'Class-weighted response',
    WEIGHTED_RESPONSE_SCALE_LIMITS,
    str_weighted_response_map,
    acc_weighted_response_map,
)

print('Selected event for class-weighted STR/ACC response map')
print(f'  path = {TARGET_EVENT_PATH.relative_to(ROOT)}')
print(f'  STR weighted response map shape = {str_weighted_response_map.shape[0]} sensors x {str_weighted_response_map.shape[1]} classes')
print(f"  STR row order = {str_weight_table['sensor_index'].astype(int).tolist()}")
print(f'  STR color scale = {WEIGHTED_RESPONSE_SCALE_LIMITS["STR"][0]:.4f} to {WEIGHTED_RESPONSE_SCALE_LIMITS["STR"][1]:.4f}')
print(f'  ACC weighted response map shape = {acc_weighted_response_map.shape[0]} sensors x {acc_weighted_response_map.shape[1]} classes')
print(f"  ACC row order = {acc_weight_table['sensor_index'].astype(int).tolist()}")
print(f'  ACC color scale = {WEIGHTED_RESPONSE_SCALE_LIMITS["ACC"][0]:.4f} to {WEIGHTED_RESPONSE_SCALE_LIMITS["ACC"][1]:.4f}')
print(f'  saved figure = {weighted_response_fig_path}')


## 3.3 ACC-Normalized STR Similarity Example

This section visualizes one representative event by decomposing the input into `STR response`, `ACC class input`, and `ACC-normalized STR response`. The map in panel (c) is the final input used to calculate `S(i,j)`.


In [ ]:
event_level_summary = (
    health_sensors
    .groupby(['path', 'deck', 'set_name', 'event_id', 'campaign_month', 'part_index'], as_index=False)
    .agg(
        median_sensor_health=('sensor_health_score', 'median'),
        median_without_acc_health=('without_acc_sensor_health_score', 'median'),
        median_sensor_time_confidence=('sensor_time_class_confidence', 'median'),
    )
)
target_event_path = str(TARGET_EVENT_PATH)
example_candidates = event_level_summary[event_level_summary['path'].eq(target_event_path)]
if example_candidates.empty:
    raise RuntimeError(f'Target event was not found in health_sensors: {target_event_path}')
example_row = example_candidates.iloc[0]
example_path = str(example_row['path'])
example_sensor_health = health_sensors[health_sensors['path'].eq(example_path)].copy()
example_ordered = example_sensor_health.set_index('sensor_alias').reindex(str_sensor_alias_order)

raw_response_map = example_ordered[CLASS_RESPONSE_COLS].to_numpy(float)
sensor_class_weight_map = np.clip(example_ordered[SENSOR_CLASS_WEIGHT_COLS].to_numpy(float), 0.0, None)
weighted_response_map = raw_response_map * np.sqrt(sensor_class_weight_map)
acc_class_input_map = example_ordered[ACC_CLASS_INPUT_COLS].to_numpy(float)
acc_normalized_response_map = _safe_divide_matrix(weighted_response_map, acc_class_input_map)
cosine_similarity, magnitude_similarity, sensor_similarity_matrix, norms = _sensor_similarity_matrices(acc_normalized_response_map)
sensor_health_vector = example_ordered['sensor_health_score'].to_numpy(float)

cmap_response = plt.get_cmap('jet').copy()
cmap_response.set_bad('#f2f2f2')
sim_cmap = plt.get_cmap('viridis').copy()
sim_cmap.set_bad('#f2f2f2')
health_cmap = plt.get_cmap('RdYlBu').copy()
health_cmap.set_bad('#f2f2f2')

fig = plt.figure(figsize=(25.0, 8.8), constrained_layout=False)
gs = fig.add_gridspec(
    2,
    5,
    height_ratios=[1.0, 0.070],
    width_ratios=[1.34, 1.34, 1.34, 1.0, 0.52],
    hspace=0.38,
    wspace=0.28,
)
axes = [fig.add_subplot(gs[0, idx]) for idx in range(5)]
cbar_axes = [fig.add_subplot(gs[1, idx]) for idx in range(5)]
fig.subplots_adjust(left=0.040, right=0.996, bottom=0.19, top=0.84)
fig.suptitle('3.2 How ACC-Classwise Input-Normalized STR Health Score Is Built', fontsize=PLOT_TITLE_FONTSIZE, y=0.96)

row_labels = str_sensor_index_labels

weighted_vmax = _finite_vmax(weighted_response_map, 99)
im0 = axes[0].imshow(weighted_response_map, aspect='auto', interpolation='nearest', cmap=cmap_response, vmin=0.0, vmax=weighted_vmax)
axes[0].set_title('(a) STR Weighted Response Map', fontsize=PANEL_TITLE_FONTSIZE)
_format_class_axis(axes[0], row_labels)
_add_bottom_colorbar(fig, cbar_axes[0], im0, 'weighted STR response')

acc_vmax = _finite_vmax(acc_class_input_map, 99)
im1 = axes[1].imshow(acc_class_input_map, aspect='auto', interpolation='nearest', cmap=cmap_response, vmin=0.0, vmax=acc_vmax)
axes[1].set_title('(b) Local ACC Class Input Map', fontsize=PANEL_TITLE_FONTSIZE)
_format_class_axis(axes[1], row_labels)
_add_bottom_colorbar(fig, cbar_axes[1], im1, 'ACC class input RMS')

norm_vmax = _finite_vmax(acc_normalized_response_map, 99)
im2 = axes[2].imshow(acc_normalized_response_map, aspect='auto', interpolation='nearest', cmap=cmap_response, vmin=0.0, vmax=norm_vmax)
axes[2].set_title('(c) STR / ACC Input Map', fontsize=PANEL_TITLE_FONTSIZE)
_format_class_axis(axes[2], row_labels)
_add_bottom_colorbar(fig, cbar_axes[2], im2, 'input-normalized STR response')

im3 = axes[3].imshow(sensor_similarity_matrix, aspect='equal', interpolation='nearest', cmap=sim_cmap, vmin=0.0, vmax=1.0)
axes[3].set_title('(d) Sensor Similarity S(i,j)', fontsize=PANEL_TITLE_FONTSIZE)
_format_similarity_axis(axes[3], row_labels)
_add_bottom_colorbar(fig, cbar_axes[3], im3, 'cosine x magnitude')

im4 = axes[4].imshow(sensor_health_vector[:, None], aspect='auto', interpolation='nearest', cmap=health_cmap, vmin=0.0, vmax=100.0)
axes[4].set_title('(e) STR Health Score', fontsize=PANEL_TITLE_FONTSIZE)
axes[4].set_xticks([0])
axes[4].set_xticklabels(['health'], fontsize=9)
axes[4].set_yticks(np.arange(len(row_labels)))
axes[4].set_yticklabels(row_labels, fontsize=8)
axes[4].set_xlabel('Score', fontsize=11, labelpad=10)
_add_bottom_colorbar(fig, cbar_axes[4], im4, 'health score')
plt.show()

print('Selected example event')
print(f"  deck = {example_row['deck']}")
print(f"  set = {example_row['set_name']}")
print(f"  part = {example_row['part_index']}")
print(f"  event_id = {example_row['event_id']}")
print(f"  ACC-input-normalized median STR health = {example_row['median_sensor_health']:.2f}")
print(f"  without-ACC median STR health = {example_row['median_without_acc_health']:.2f}")
print()
print('Pipeline: weighted STR response -> divide by same span/side ACC class input RMS -> S(i,j) -> STR health score.')

example_display_columns = [
    'sensor_index', 'sensor_alias', 'local_group_id', 'comparison_group', 'span', 'location',
    *ACC_CLASS_INPUT_COLS,
    'same_group_S_avg', 'opposite_span_S_avg', 'without_acc_sensor_health_score', 'sensor_health_score',
]
display(example_ordered.reset_index()[example_display_columns])

## 3.4 From S(i,j) to STR Health Score

The next figures use the ACC-normalized sensor similarity matrix from the previous section. Each comparison group is shown separately so that the same-group and opposite-span averages can be read directly from one group block.


In [ ]:
group_plot_table = str_meta_ordered.copy()
visible_group_order = ['STR DO', 'STR UP']
visible_group_order = [group for group in visible_group_order if group in set(group_plot_table['comparison_group'])]
group_rank_lookup = {group: idx for idx, group in enumerate(visible_group_order)}
group_plot_table['group_rank'] = group_plot_table['comparison_group'].map(group_rank_lookup).astype(int)
group_plot_table['span_rank'] = group_plot_table['span'].map(SPAN_ORDER).fillna(99).astype(int)
group_plot_table['location_rank'] = group_plot_table['location'].map(LOCATION_ORDER).fillna(99).astype(int)
group_plot_table = group_plot_table.sort_values(['group_rank', 'span_rank', 'location_rank', 'sensor_index']).reset_index(drop=True)

ordered_aliases = group_plot_table['sensor_alias'].tolist()
ordered_matrix_indices = [str_sensor_alias_order.index(alias) for alias in ordered_aliases]
ordered_meta = group_plot_table.set_index('sensor_alias')
label_lookup = group_plot_table.set_index('sensor_alias')['sensor_index'].to_dict()
health_lookup = example_ordered['sensor_health_score'].to_dict()
same_avg_lookup = example_ordered['same_group_S_avg'].to_dict()
opposite_avg_lookup = example_ordered['opposite_span_S_avg'].to_dict()

row_specs = []
for group in visible_group_order:
    local_aliases = group_plot_table.loc[group_plot_table['comparison_group'].eq(group), 'sensor_alias'].tolist()
    row_specs.append({
        'group': group,
        'group_id': group_plot_table.loc[group_plot_table['comparison_group'].eq(group), 'local_group_id'].iloc[0],
        'local_aliases': local_aliases,
        'focus_alias': local_aliases[0],
    })

ordered_similarity = sensor_similarity_matrix[np.ix_(ordered_matrix_indices, ordered_matrix_indices)]
guide_color = '#ffffff'
active_tick_color = '#111111'
inactive_tick_color = '#9a9a9a'
panel_titles = ['(a) Input S(i,j) block', '(b) Same-group cells averaged', '(c) Opposite-span cells averaged']


def _add_span_boundaries(ax, local_spans, color='white'):
    local_spans = np.asarray(local_spans, dtype=str)
    change_idx = np.flatnonzero(local_spans[:-1] != local_spans[1:])
    for idx in change_idx:
        boundary = idx + 0.5
        ax.axhline(boundary, color=color, lw=2.0, alpha=0.95, zorder=4)
        ax.axvline(boundary, color=color, lw=2.0, alpha=0.95, zorder=4)


def _add_focus_average_guides(ax, focus_idx, active_cols, color, label):
    n = len(active_cols)
    # The score for a displayed focus sensor uses only this focus row.
    ax.add_patch(Rectangle((-0.5, focus_idx - 0.5), n, 1, fill=False, edgecolor=color, linewidth=2.2, linestyle='--', zorder=5, clip_on=False))
    for col_idx, is_active in enumerate(active_cols):
        if is_active:
            ax.add_patch(Rectangle((col_idx - 0.5, focus_idx - 0.5), 1, 1, fill=False, edgecolor=color, linewidth=2.8, linestyle='-', zorder=6, clip_on=False))
    ax.add_patch(Rectangle((focus_idx - 0.5, focus_idx - 0.5), 1, 1, fill=False, edgecolor='#262626', linewidth=3.0, zorder=7, clip_on=False))
    ax.text(
        0.03,
        0.04,
        label,
        transform=ax.transAxes,
        fontsize=10.2,
        ha='left',
        va='bottom',
        color='white',
        bbox=dict(facecolor='black', alpha=0.55, edgecolor='none', boxstyle='round,pad=0.18'),
        clip_on=False,
    )


def _add_average_fade_overlay(ax, active_cell_mask, alpha=0.68):
    fade_overlay = np.ones((*active_cell_mask.shape, 4), dtype=float)
    fade_overlay[..., :3] = 1.0
    fade_overlay[..., 3] = np.where(active_cell_mask, 0.0, alpha)
    ax.imshow(fade_overlay, aspect='equal', interpolation='nearest', zorder=2)


def _format_group_block_axis(ax, local_aliases, active_x_mask=None, active_y_mask=None):
    labels = [str(int(label_lookup[alias])) for alias in local_aliases]
    ax.set_xticks(np.arange(len(local_aliases)))
    ax.set_yticks(np.arange(len(local_aliases)))
    ax.set_xticklabels(labels, fontsize=12.0)
    ax.set_yticklabels(labels, fontsize=12.0)
    ax.tick_params(length=2.0, pad=2.0)
    ax.set_xlabel('Sensor j index', fontsize=12.0, labelpad=7.0)
    ax.set_ylabel('Sensor i index', fontsize=12.0)
    if active_x_mask is not None:
        for tick_label, is_active in zip(ax.get_xticklabels(), active_x_mask):
            tick_label.set_color(active_tick_color if is_active else inactive_tick_color)
            tick_label.set_fontweight('bold' if is_active else 'normal')
    if active_y_mask is not None:
        for tick_label, is_active in zip(ax.get_yticklabels(), active_y_mask):
            tick_label.set_color(active_tick_color if is_active else inactive_tick_color)
            tick_label.set_fontweight('bold' if is_active else 'normal')


def _plot_selected_group_blocks_from_full_similarity_map():
    full_labels = [str(int(label_lookup[alias])) for alias in ordered_aliases]
    selected_block_mask = np.zeros_like(ordered_similarity, dtype=bool)
    group_ranges = []
    start = 0
    for spec in row_specs:
        count = len(spec['local_aliases'])
        group_ranges.append((spec, start, count))
        selected_block_mask[start:start + count, start:start + count] = True
        start += count

    fig, (ax_info, ax) = plt.subplots(
        1,
        2,
        figsize=(11.6, 7.2),
        constrained_layout=False,
        gridspec_kw={'width_ratios': [0.52, 1.48]},
    )
    fig.subplots_adjust(left=0.045, right=0.965, bottom=0.155, top=0.850, wspace=0.070)
    fig.suptitle(
        '3.3a Selected STR Blocks from Inter-Sensor Spectral Response Similarity Map, S(i,j)',
        fontsize=PLOT_TITLE_FONTSIZE + 1,
        y=0.955,
    )

    ax_info.axis('off')
    ax_info.set_xlim(0.0, 1.0)
    ax_info.set_ylim(0.0, 1.0)
    ax_info.text(
        0.02,
        0.92,
        'Selected\nSTR blocks',
        ha='left',
        va='top',
        fontsize=15.0,
        fontweight='bold',
    )

    for idx, (spec, block_start, count) in enumerate(group_ranges):
        y_top = 0.68 - idx * 0.32
        sensor_indices = [str(int(label_lookup[alias])) for alias in spec['local_aliases']]
        ax_info.add_patch(Rectangle(
            (0.03, y_top - 0.050),
            0.095,
            0.095,
            fill=False,
            edgecolor='black',
            linewidth=2.0,
            clip_on=False,
        ))
        ax_info.text(
            0.16,
            y_top,
            f"{spec['group_id']} {spec['group']}",
            ha='left',
            va='center',
            fontsize=13.0,
            fontweight='bold',
        )
        ax_info.text(
            0.16,
            y_top - 0.085,
            'selected 6 x 6 block\n' + 'sensor indices: ' + ', '.join(sensor_indices),
            ha='left',
            va='top',
            fontsize=10.6,
            color='0.18',
            linespacing=1.35,
        )

    ax_info.text(
        0.02,
        0.105,
        'The following panels zoom into\neach selected same-group block.',
        ha='left',
        va='bottom',
        fontsize=10.4,
        color='0.25',
        linespacing=1.35,
    )

    im = ax.imshow(ordered_similarity, aspect='equal', interpolation='nearest', cmap=sim_cmap, vmin=0.0, vmax=1.0, zorder=1)
    fade_overlay = np.ones((*ordered_similarity.shape, 4), dtype=float)
    fade_overlay[..., :3] = 1.0
    fade_overlay[..., 3] = np.where(selected_block_mask, 0.0, 0.58)
    ax.imshow(fade_overlay, aspect='equal', interpolation='nearest', zorder=2)

    ax.set_xticks(np.arange(len(ordered_aliases)))
    ax.set_yticks(np.arange(len(ordered_aliases)))
    ax.set_xticklabels(full_labels, fontsize=11.0)
    ax.set_yticklabels(full_labels, fontsize=11.0)
    ax.set_xlabel('Sensor j index', fontsize=12.5, labelpad=7.0)
    ax.set_ylabel('Sensor i index', fontsize=12.5)
    ax.tick_params(length=2.2, pad=2.0)

    for spec, block_start, count in group_ranges:
        ax.add_patch(Rectangle(
            (block_start - 0.5, block_start - 0.5),
            count,
            count,
            fill=False,
            edgecolor='white',
            linewidth=3.2,
            zorder=4,
            clip_on=False,
        ))
        ax.add_patch(Rectangle(
            (block_start - 0.5, block_start - 0.5),
            count,
            count,
            fill=False,
            edgecolor='black',
            linewidth=1.2,
            zorder=5,
            clip_on=False,
        ))

    ax.add_patch(Rectangle(
        (-0.5, -0.5),
        len(ordered_aliases),
        len(ordered_aliases),
        fill=False,
        edgecolor='black',
        linewidth=2.0,
        zorder=6,
        clip_on=False,
    ))

    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.055, pad=0.110)
    cbar.set_label('S(i,j) = cosine similarity x normalized magnitude similarity', fontsize=10.5)
    _set_colorbar_min_max_ticks(cbar, im=im)
    cbar.ax.tick_params(labelsize=9.5, length=2.4, pad=1.4)

    fig.text(
        0.60,
        0.045,
        'Only the highlighted same-group STR blocks are used as the 6x6 input S(i,j) blocks in the following panels.',
        ha='center',
        fontsize=10.8,
    )
    selection_fig_path = FIG_DIR / '3_3a_selected_str_blocks_from_inter_sensor_similarity_map.png'
    fig.savefig(selection_fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'saved figure = {selection_fig_path}')


_plot_selected_group_blocks_from_full_similarity_map()


summary_rows = []
for spec in row_specs:
    group = spec['group']
    group_id = spec['group_id']
    local_aliases = spec['local_aliases']
    focus_alias = spec['focus_alias']
    focus_idx = local_aliases.index(focus_alias)
    local_indices = [ordered_aliases.index(a) for a in local_aliases]
    local_matrix = ordered_similarity[np.ix_(local_indices, local_indices)]
    local_spans = np.array([ordered_meta.loc[a, 'span'] for a in local_aliases], dtype=str)
    same_group_pair_mask = ~np.eye(len(local_aliases), dtype=bool)
    opposite_span_pair_mask = local_spans[:, None] != local_spans[None, :]
    same_focus_cols = same_group_pair_mask[focus_idx]
    opposite_focus_cols = opposite_span_pair_mask[focus_idx]
    focus_sensor_index = int(label_lookup[focus_alias])
    focus_mask = np.eye(len(local_aliases), dtype=bool)[focus_idx]
    same_focus_cell_mask = np.zeros_like(local_matrix, dtype=bool)
    same_focus_cell_mask[focus_idx, same_focus_cols] = True
    opposite_focus_cell_mask = np.zeros_like(local_matrix, dtype=bool)
    opposite_focus_cell_mask[focus_idx, opposite_focus_cols] = True
    same_focus_or_peer = same_focus_cols | focus_mask
    focus_or_opposite = opposite_focus_cols | focus_mask
    same_avg = float(same_avg_lookup[focus_alias])
    opposite_avg = float(opposite_avg_lookup[focus_alias])
    health_score = float(health_lookup[focus_alias])

    fig, axes = plt.subplots(1, 3, figsize=(17.4, 6.25), constrained_layout=False)
    fig.subplots_adjust(left=0.070, right=0.975, bottom=0.285, top=0.800, wspace=0.26)
    fig.suptitle(
        f'3.3 {group_id} {group}: S(i,j) to Same/Opposite Averages to Health Score',
        fontsize=PLOT_TITLE_FONTSIZE + 2,
        y=0.955,
    )

    im = axes[0].imshow(local_matrix, aspect='equal', interpolation='nearest', cmap=sim_cmap, vmin=0.0, vmax=1.0, zorder=1)
    axes[0].set_title(panel_titles[0], fontsize=PANEL_TITLE_FONTSIZE + 2, pad=9)
    _format_group_block_axis(axes[0], local_aliases)
    _add_span_boundaries(axes[0], local_spans, color='white')
    axes[0].add_patch(Rectangle((-0.5, -0.5), len(local_aliases), len(local_aliases), fill=False, edgecolor='black', linewidth=2.4, zorder=5, clip_on=False))

    axes[1].imshow(local_matrix, aspect='equal', interpolation='nearest', cmap=sim_cmap, vmin=0.0, vmax=1.0, zorder=1)
    _add_average_fade_overlay(axes[1], same_focus_cell_mask, alpha=0.68)
    axes[1].set_title(panel_titles[1], fontsize=PANEL_TITLE_FONTSIZE + 2, pad=9)
    _format_group_block_axis(axes[1], local_aliases, active_x_mask=same_focus_or_peer, active_y_mask=focus_mask)
    _add_span_boundaries(axes[1], local_spans, color='white')
    axes[1].add_patch(Rectangle((-0.5, -0.5), len(local_aliases), len(local_aliases), fill=False, edgecolor='black', linewidth=2.4, zorder=5, clip_on=False))
    _add_focus_average_guides(axes[1], focus_idx, same_focus_cols, guide_color, f'focus = {focus_sensor_index}')

    axes[2].imshow(local_matrix, aspect='equal', interpolation='nearest', cmap=sim_cmap, vmin=0.0, vmax=1.0, zorder=1)
    _add_average_fade_overlay(axes[2], opposite_focus_cell_mask, alpha=0.68)
    axes[2].set_title(panel_titles[2], fontsize=PANEL_TITLE_FONTSIZE + 2, pad=9)
    _format_group_block_axis(axes[2], local_aliases, active_x_mask=focus_or_opposite, active_y_mask=focus_mask)
    _add_span_boundaries(axes[2], local_spans, color='white')
    axes[2].add_patch(Rectangle((-0.5, -0.5), len(local_aliases), len(local_aliases), fill=False, edgecolor='black', linewidth=2.4, zorder=5, clip_on=False))
    _add_focus_average_guides(axes[2], focus_idx, opposite_focus_cols, guide_color, f'focus = {focus_sensor_index}')

    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), orientation='horizontal', fraction=0.055, pad=0.185)
    cbar.set_label('S(i,j) = cosine similarity x normalized magnitude similarity', fontsize=11)
    _set_colorbar_min_max_ticks(cbar, im=im)
    cbar.ax.tick_params(labelsize=10, length=2.4, pad=1.4)
    fig.text(
        0.50,
        0.090,
        f'Focus sensor {focus_sensor_index}: only highlighted cells in the focus row are averaged; self-pair S(i,i) is excluded.',
        ha='center',
        fontsize=12.0,
    )
    plt.show()

    summary_rows.append({
        'group_id': group_id,
        'comparison_group': group,
        'focus_sensor_index': focus_sensor_index,
        'focus_sensor_alias': focus_alias,
        'same_group_S_avg': same_avg,
        'opposite_span_S_avg': opposite_avg,
        'sensor_health_score': health_score,
    })

print('Each figure uses only the ACC-normalized S(i,j) block for one STR comparison group.')
print('Tick labels are method-level sensor indices only; aliases are kept in the summary table.')
print('Self-pair S(i,i) is boxed but excluded from the average.')
display(pd.DataFrame(summary_rows))


## 3.5 Sensor Health Score Trend and SD Reduction

This section shows the G1/G2 STR score trends using a sensor-wise subplot grid. Each subplot corresponds to one STR sensor. The solid line represents the ACC-classwise input-normalized score, while the dashed line represents the score without ACC input normalization. The summary table compares the score standard deviation before and after ACC normalization.


In [ ]:
trend_plot_table = str_sensor_meta.copy()
trend_plot_table['trend_group'] = trend_plot_table['comparison_group']
trend_plot_table['trend_group_id'] = trend_plot_table['local_group_id']
local_group_order = ['STR DO', 'STR UP']
_group_row_lookup = {group: idx for idx, group in enumerate(local_group_order)}
trend_plot_table['group_plot_row'] = trend_plot_table['trend_group'].map(_group_row_lookup).astype(int)
trend_plot_table['location_rank'] = trend_plot_table['location'].map(LOCATION_ORDER).fillna(99).astype(int)
trend_plot_table['span_rank'] = trend_plot_table['span'].map(SPAN_ORDER).fillna(99).astype(int)
trend_plot_table['axis_rank'] = 0
trend_plot_table = trend_plot_table.sort_values([
    'group_plot_row',
    'span_rank',
    'location_rank',
    'axis_rank',
    'sensor_index',
]).reset_index(drop=True)
trend_plot_table['group_plot_col'] = trend_plot_table.groupby('trend_group').cumcount()

N_PLOT_ROWS = int(trend_plot_table['group_plot_row'].max() + 1)
N_PLOT_COLS = int(trend_plot_table['group_plot_col'].max() + 1)
AXIS_LABEL_FONTSIZE = 17
TICK_LABEL_FONTSIZE = 11
PANEL_LABEL_FONTSIZE = 12.5


def _sensor_short_label(row):
    axis = str(row['axis']) if pd.notna(row['axis']) and str(row['axis']) != 'nan' else ''
    axis_text = f'-{axis}' if axis else ''
    return f"{row['span']}-{row['side']}-{row['location']} {row['quantity']}{axis_text}"


def _low_frequency_health_trend(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return values
    window = max(31, min(401, int(len(values) * 0.06) // 2 * 2 + 1))
    return pd.Series(values).rolling(
        window=window,
        center=True,
        min_periods=max(5, window // 8),
    ).median().to_numpy(float)

trend_cache = {}
baseline_rows = []
for _, meta_row in trend_plot_table.iterrows():
    sensor_alias = meta_row['sensor_alias']
    for deck in ['OLD', 'NEW']:
        d = health_sensors[
            health_sensors['deck'].eq(deck)
            & health_sensors['sensor_alias'].eq(sensor_alias)
        ].sort_values('event_sequence').copy()
        x = d['event_sequence'].to_numpy(float)
        raw_y = d['sensor_health_score'].to_numpy(float)
        raw_y_without = d['without_acc_sensor_health_score'].to_numpy(float)
        finite = np.isfinite(raw_y)
        finite_without = np.isfinite(raw_y_without)
        if finite.sum() == 0:
            raw_trend = np.full_like(raw_y, np.nan, dtype=float)
            sensor_baseline = np.nan
        else:
            raw_trend = _low_frequency_health_trend(raw_y)
            sensor_baseline = float(np.nanmedian(raw_trend)) if np.isfinite(raw_trend).any() else np.nan
        if finite_without.sum() == 0:
            raw_without_trend = np.full_like(raw_y_without, np.nan, dtype=float)
            sensor_without_baseline = np.nan
        else:
            raw_without_trend = _low_frequency_health_trend(raw_y_without)
            sensor_without_baseline = float(np.nanmedian(raw_without_trend)) if np.isfinite(raw_without_trend).any() else np.nan
        trend_cache[(deck, sensor_alias)] = {
            'x': x,
            'raw_y': raw_y,
            'raw_y_without': raw_y_without,
            'finite': finite,
            'finite_without': finite_without,
            'raw_trend': raw_trend,
            'raw_without_trend': raw_without_trend,
            'sensor_baseline': sensor_baseline,
            'sensor_without_baseline': sensor_without_baseline,
            'trend_group': meta_row['trend_group'],
        }
        baseline_rows.extend([
            {
                'deck': deck,
                'sensor_alias': sensor_alias,
                'trend_group': meta_row['trend_group'],
                'mode': 'with_acc',
                'sensor_baseline': sensor_baseline,
            },
            {
                'deck': deck,
                'sensor_alias': sensor_alias,
                'trend_group': meta_row['trend_group'],
                'mode': 'without_acc',
                'sensor_baseline': sensor_without_baseline,
            },
        ])

baseline_table = pd.DataFrame(baseline_rows)
group_baseline_lookup = baseline_table.groupby(['deck', 'trend_group', 'mode'])['sensor_baseline'].median().to_dict()

for _, meta_row in trend_plot_table.iterrows():
    sensor_alias = meta_row['sensor_alias']
    for deck in ['OLD', 'NEW']:
        item = trend_cache[(deck, sensor_alias)]
        group_baseline = float(group_baseline_lookup.get((deck, meta_row['trend_group'], 'with_acc'), np.nan))
        sensor_baseline = item['sensor_baseline']
        shift = 0.0 if not np.isfinite(sensor_baseline) or not np.isfinite(group_baseline) else group_baseline - sensor_baseline

        group_without_baseline = float(group_baseline_lookup.get((deck, meta_row['trend_group'], 'without_acc'), np.nan))
        sensor_without_baseline = item['sensor_without_baseline']
        shift_without = 0.0 if not np.isfinite(sensor_without_baseline) or not np.isfinite(group_without_baseline) else group_without_baseline - sensor_without_baseline

        item['plot_y'] = item['raw_y'] + shift
        item['plot_y_without'] = item['raw_y_without'] + shift_without
        item['plot_trend'] = item['raw_trend'] + shift
        item['plot_without_trend'] = item['raw_without_trend'] + shift_without
        item['baseline_shift'] = shift
        item['without_baseline_shift'] = shift_without
        item['display_trend'] = item['plot_trend']
        item['display_without_trend'] = item['plot_without_trend']

sensor_trend_rows = []
for _, meta_row in trend_plot_table.iterrows():
    sensor_alias = meta_row['sensor_alias']
    sensor_index = int(meta_row['sensor_index'])
    for deck in ['OLD', 'NEW']:
        item = trend_cache[(deck, sensor_alias)]
        y = item['plot_y']
        y_without = item['plot_y_without']
        trend = item['display_trend']
        trend_without = item['display_without_trend']
        finite = np.isfinite(y)
        finite_without = np.isfinite(y_without)
        trend_start = float(trend[0]) if len(trend) and np.isfinite(trend[0]) else np.nan
        trend_end = float(trend[-1]) if len(trend) and np.isfinite(trend[-1]) else np.nan
        trend_delta = float(trend_end - trend_start) if np.isfinite(trend_start) and np.isfinite(trend_end) else np.nan
        trend_without_start = float(trend_without[0]) if len(trend_without) and np.isfinite(trend_without[0]) else np.nan
        trend_without_end = float(trend_without[-1]) if len(trend_without) and np.isfinite(trend_without[-1]) else np.nan
        trend_without_delta = float(trend_without_end - trend_without_start) if np.isfinite(trend_without_start) and np.isfinite(trend_without_end) else np.nan
        sensor_trend_rows.append({
            'deck': deck,
            'sensor_index': sensor_index,
            'sensor_alias': sensor_alias,
            'local_group_id': meta_row['trend_group_id'],
            'comparison_group': meta_row['trend_group'],
            'quantity': meta_row['quantity'],
            'span': meta_row['span'],
            'side': meta_row['side'],
            'location': meta_row['location'],
            'axis': meta_row['axis'],
            'n_events': int(finite.sum()),
            'without_acc_n_events': int(finite_without.sum()),
            'health_median': float(np.nanmedian(y)) if finite.sum() else np.nan,
            'without_acc_health_median': float(np.nanmedian(y_without)) if finite_without.sum() else np.nan,
            'baseline_shift': float(item['baseline_shift']),
            'without_acc_baseline_shift': float(item['without_baseline_shift']),
            'trend_start': trend_start,
            'trend_end': trend_end,
            'trend_end_minus_start': trend_delta,
            'without_acc_trend_end_minus_start': trend_without_delta,
        })

sensor_trend_summary = pd.DataFrame(sensor_trend_rows)
group_delta_sd = sensor_trend_summary.groupby(['deck', 'comparison_group'])['trend_end_minus_start'].std(ddof=0).fillna(0.0).to_dict()
group_without_delta_sd = sensor_trend_summary.groupby(['deck', 'comparison_group'])['without_acc_trend_end_minus_start'].std(ddof=0).fillna(0.0).to_dict()
sensor_delta_lookup = sensor_trend_summary.set_index(['deck', 'sensor_alias'])['trend_end_minus_start'].to_dict()
sensor_without_delta_lookup = sensor_trend_summary.set_index(['deck', 'sensor_alias'])['without_acc_trend_end_minus_start'].to_dict()

month_tick_source = health_sensors[['campaign_month', 'event_sequence']].dropna().copy()
month_tick_source['campaign_month_dt'] = pd.to_datetime(month_tick_source['campaign_month'].astype(str), errors='coerce')
month_tick_table = (
    month_tick_source.dropna(subset=['campaign_month_dt'])
    .groupby('campaign_month_dt', as_index=False)['event_sequence']
    .median()
    .sort_values('campaign_month_dt')
)
MONTH_TICK_POSITIONS = month_tick_table['event_sequence'].to_numpy(float)
MONTH_TICK_LABELS = [f'{dt.year}\n{dt.month:02d}' for dt in month_tick_table['campaign_month_dt']]
EVENT_SEQUENCE_LIMITS = (
    float(np.nanmin(health_sensors['event_sequence'])),
    float(np.nanmax(health_sensors['event_sequence'])),
)


def _group_zoom_ylim(trend_group, trend_key):
    values = []
    for _, row in trend_plot_table[trend_plot_table['trend_group'].eq(trend_group)].iterrows():
        for deck in ['OLD', 'NEW']:
            trend = np.asarray(trend_cache[(deck, row['sensor_alias'])][trend_key], dtype=float)
            if np.isfinite(trend).any():
                values.append(trend[np.isfinite(trend)])
    if not values:
        return 48.0, 102.0
    flat = np.concatenate(values)
    y_low = float(np.nanmin(flat))
    y_high = float(np.nanmax(flat))
    span = y_high - y_low
    if span < 2.0:
        center = 0.5 * (y_low + y_high)
        y_low = center - 1.0
        y_high = center + 1.0
        span = y_high - y_low
    y_pad = max(0.60, 0.22 * span)
    return max(0.0, y_low - y_pad), min(100.0, y_high + y_pad)


def _plot_sensor_trend_grid(title, y_key, trend_key, delta_lookup, sd_lookup):
    fig, axes = plt.subplots(N_PLOT_ROWS, N_PLOT_COLS, figsize=(26.0, 8.8), sharex=False, sharey=False, constrained_layout=False)
    if N_PLOT_ROWS == 1:
        axes = np.asarray([axes])
    fig.subplots_adjust(left=0.060, right=0.995, bottom=0.115, top=0.865, wspace=0.18, hspace=0.82)
    fig.suptitle(title, fontsize=PLOT_TITLE_FONTSIZE + 4, y=0.975)

    used_axes = set()
    for _, meta_row in trend_plot_table.iterrows():
        row_idx = int(meta_row['group_plot_row'])
        col_idx = int(meta_row['group_plot_col'])
        ax = axes[row_idx, col_idx]
        used_axes.add((row_idx, col_idx))
        sensor_alias = meta_row['sensor_alias']

        for deck in ['OLD', 'NEW']:
            item = trend_cache.get((deck, sensor_alias))
            if item is None:
                continue
            x = item['x']
            y = item[y_key]
            finite = np.isfinite(y)
            trend = item[trend_key]
            if finite.sum() == 0:
                continue
            ax.scatter(x[finite], y[finite], s=1.5, color=DECK_COLORS[deck], alpha=0.040, linewidths=0, rasterized=True)
            ax.plot(x, trend, color=DECK_COLORS[deck], lw=2.05)

        ax.set_title(_sensor_short_label(meta_row), fontsize=PANEL_LABEL_FONTSIZE, pad=5)
        ax.set_xlim(*EVENT_SEQUENCE_LIMITS)
        ax.set_ylim(*_group_zoom_ylim(meta_row['trend_group'], trend_key))
        ax.grid(True, alpha=0.22, linewidth=0.6)
        if len(MONTH_TICK_POSITIONS):
            ax.set_xticks(MONTH_TICK_POSITIONS)
            if row_idx == N_PLOT_ROWS - 1:
                ax.set_xticklabels(MONTH_TICK_LABELS, fontsize=TICK_LABEL_FONTSIZE)
            else:
                ax.set_xticklabels([])
        ax.tick_params(axis='x', labelsize=TICK_LABEL_FONTSIZE, length=2.8, pad=2.0)
        ax.tick_params(axis='y', labelsize=TICK_LABEL_FONTSIZE, length=2.8, pad=1.8)
        if col_idx == 0:
            ax.set_ylabel('Health score', fontsize=14.0)

    for row_idx in range(N_PLOT_ROWS):
        for col_idx in range(N_PLOT_COLS):
            if (row_idx, col_idx) not in used_axes:
                axes[row_idx, col_idx].axis('off')

    plt.show()


_plot_sensor_trend_grid(
    '3.4a STR Sensor Health Score Trends With ACC Input Normalization',
    'plot_y',
    'display_trend',
    sensor_delta_lookup,
    group_delta_sd,
)

_plot_sensor_trend_grid(
    '3.4b STR Sensor Health Score Trends Without ACC Input Normalization',
    'plot_y_without',
    'display_without_trend',
    sensor_without_delta_lookup,
    group_without_delta_sd,
)

event_group_summary = (
    health_sensors
    .groupby(['path', 'deck', 'campaign_month', 'event_sequence', 'local_group_id', 'comparison_group'], as_index=False)
    .agg(
        event_group_median_score=('sensor_health_score', 'median'),
        event_group_median_without_acc=('without_acc_sensor_health_score', 'median'),
        event_group_sensor_sd=('sensor_health_score', 'std'),
        event_group_without_acc_sensor_sd=('without_acc_sensor_health_score', 'std'),
    )
)

sd_summary = event_group_summary.groupby(['local_group_id', 'comparison_group', 'deck']).agg(
    event_count=('event_group_median_score', 'size'),
    with_acc_event_median_score=('event_group_median_score', 'median'),
    without_acc_event_median_score=('event_group_median_without_acc', 'median'),
    with_acc_event_score_sd=('event_group_median_score', 'std'),
    without_acc_event_score_sd=('event_group_median_without_acc', 'std'),
    with_acc_within_event_sensor_sd_median=('event_group_sensor_sd', 'median'),
    without_acc_within_event_sensor_sd_median=('event_group_without_acc_sensor_sd', 'median'),
).round(6)
sd_summary['event_score_sd_reduction'] = sd_summary['without_acc_event_score_sd'] - sd_summary['with_acc_event_score_sd']
sd_summary['within_event_sensor_sd_reduction'] = sd_summary['without_acc_within_event_sensor_sd_median'] - sd_summary['with_acc_within_event_sensor_sd_median']
print('Event-level G1/G2 score SD comparison: with ACC vs without ACC')
display(sd_summary.round(6))

sensor_level_summary = health_sensors.groupby(['local_group_id', 'comparison_group', 'deck']).agg(
    with_acc_sensor_level_sd=('sensor_health_score', 'std'),
    without_acc_sensor_level_sd=('without_acc_sensor_health_score', 'std'),
    with_acc_sensor_level_median=('sensor_health_score', 'median'),
    without_acc_sensor_level_median=('without_acc_sensor_health_score', 'median'),
).round(6)
sensor_level_summary['sensor_level_sd_reduction'] = sensor_level_summary['without_acc_sensor_level_sd'] - sensor_level_summary['with_acc_sensor_level_sd']
print('Sensor-level G1/G2 score SD comparison')
display(sensor_level_summary.round(6))

print('Key NEW result')
for group_id, group_name in [('G1', 'STR DO'), ('G2', 'STR UP')]:
    row = sd_summary.loc[(group_id, group_name, 'NEW')]
    print(f"  {group_id} {group_name} NEW event-score SD: without ACC {row['without_acc_event_score_sd']:.3f} -> with ACC {row['with_acc_event_score_sd']:.3f}")

# display(sensor_trend_summary.sort_values(['local_group_id', 'location', 'span', 'deck']).reset_index(drop=True))

## 3.6 Final Group-Level STR Health Score Trend

The six STR sensor-level health score traces in each comparison group are averaged into one group-level event trend for NEW and OLD decks.


In [ ]:
final_group_order = [
    ('G1', 'STR DO'),
    ('G2', 'STR UP'),
]

if '_low_frequency_health_trend' not in globals():
    def _low_frequency_health_trend(values):
        values = np.asarray(values, dtype=float)
        if len(values) == 0:
            return values
        window = max(31, min(401, int(len(values) * 0.06) // 2 * 2 + 1))
        return pd.Series(values).rolling(
            window=window,
            center=True,
            min_periods=max(5, window // 8),
        ).median().to_numpy(float)

final_group_event_trend = (
    health_sensors
    .groupby(['deck', 'event_sequence', 'local_group_id', 'comparison_group'], as_index=False)
    .agg(
        group_mean_health=('sensor_health_score', 'mean'),
        group_median_health=('sensor_health_score', 'median'),
        group_sensor_sd=('sensor_health_score', 'std'),
        sensor_count=('sensor_alias', 'nunique'),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(17.2, 5.8), constrained_layout=False, sharey=True)
fig.subplots_adjust(left=0.070, right=0.990, bottom=0.280, top=0.790, wspace=0.18)
fig.suptitle('3.5 Final Group-Level ACC-Normalized STR Health Score Trends', fontsize=PLOT_TITLE_FONTSIZE, y=0.935)

summary_rows = []
for ax, (group_id, group_name) in zip(axes, final_group_order):
    for deck in ['NEW', 'OLD']:
        d = final_group_event_trend[
            final_group_event_trend['deck'].eq(deck)
            & final_group_event_trend['local_group_id'].eq(group_id)
            & final_group_event_trend['comparison_group'].eq(group_name)
        ].sort_values('event_sequence').copy()

        color = DECK_COLORS[deck]
        x = d['event_sequence'].to_numpy(float)
        y = d['group_mean_health'].to_numpy(float)
        finite = np.isfinite(y)
        trend = _low_frequency_health_trend(y)

        ax.scatter(x[finite], y[finite], s=4.0, color=color, alpha=0.075, linewidths=0, rasterized=True)
        ax.plot(x, trend, color=color, lw=2.55, label=deck)

        summary_rows.append({
            'local_group_id': group_id,
            'comparison_group': group_name,
            'deck': deck,
            'event_count': int(len(d)),
            'sensor_count_per_event': int(d['sensor_count'].median()) if len(d) else 0,
            'mean_health_median': float(np.nanmedian(y)) if finite.any() else np.nan,
            'mean_health_sd': float(np.nanstd(y, ddof=1)) if finite.sum() > 1 else np.nan,
            'trend_start': float(trend[np.isfinite(trend)][0]) if np.isfinite(trend).any() else np.nan,
            'trend_end': float(trend[np.isfinite(trend)][-1]) if np.isfinite(trend).any() else np.nan,
        })

    ax.set_title(f'{group_id} {group_name}\nmean of 6 STR sensors', fontsize=PANEL_TITLE_FONTSIZE, pad=8)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.24, linewidth=0.6)
    ax.tick_params(axis='both', labelsize=9.0, length=2.5, pad=1.5)
    ax.set_xlabel('Event sequence', fontsize=10)

axes[0].set_ylabel('Mean STR health score', fontsize=10)

legend_handles = [
    Line2D([0], [0], color=DECK_COLORS['NEW'], lw=2.8, label='NEW'),
    Line2D([0], [0], color=DECK_COLORS['OLD'], lw=2.8, label='OLD'),
]
fig.text(
    0.50,
    0.160,
    'Each panel averages the six STR sensor health score traces within the corresponding comparison group.',
    ha='center',
    fontsize=10.5,
)
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.055),
    ncol=2,
    frameon=True,
    fontsize=11,
)
plt.show()

final_group_trend_summary = pd.DataFrame(summary_rows)
print('Final group-level trend summary')
display(final_group_trend_summary.round(3))

## 3.7 Overall STR Health Score Trend

The final figure combines G1 and G2 into one event-level STR health score trend for each deck. Sensor-specific median level differences are aligned before averaging so the time-dependent trend is not diluted by static sensor offsets.


In [ ]:
if '_low_frequency_health_trend' not in globals():
    def _low_frequency_health_trend(values):
        values = np.asarray(values, dtype=float)
        if len(values) == 0:
            return values
        window = max(31, min(401, int(len(values) * 0.06) // 2 * 2 + 1))
        return pd.Series(values).rolling(
            window=window,
            center=True,
            min_periods=max(5, window // 8),
        ).median().to_numpy(float)

# Align sensor-specific median levels before averaging.
# This is a visualization normalization only; the health score values in health_sensors are not changed.
overall_sensor_cache = {}
overall_baseline_rows = []
for (deck, sensor_alias), d in health_sensors.groupby(['deck', 'sensor_alias'], sort=False):
    d = d.sort_values('event_sequence').copy()
    x = d['event_sequence'].to_numpy(float)
    raw_y = d['sensor_health_score'].to_numpy(float)
    finite = np.isfinite(raw_y)
    raw_trend = _low_frequency_health_trend(raw_y) if finite.sum() else np.full_like(raw_y, np.nan, dtype=float)
    sensor_baseline = float(np.nanmedian(raw_trend)) if np.isfinite(raw_trend).any() else np.nan
    overall_sensor_cache[(deck, sensor_alias)] = {
        'x': x,
        'raw_y': raw_y,
        'raw_trend': raw_trend,
        'sensor_baseline': sensor_baseline,
        'event_sequence': d['event_sequence'].to_numpy(int),
    }
    overall_baseline_rows.append({
        'deck': deck,
        'sensor_alias': sensor_alias,
        'sensor_baseline': sensor_baseline,
    })

overall_baseline_table = pd.DataFrame(overall_baseline_rows)
overall_deck_baseline_lookup = overall_baseline_table.groupby('deck')['sensor_baseline'].median().to_dict()

aligned_rows = []
for (deck, sensor_alias), item in overall_sensor_cache.items():
    deck_baseline = float(overall_deck_baseline_lookup.get(deck, np.nan))
    sensor_baseline = item['sensor_baseline']
    shift = 0.0 if not np.isfinite(deck_baseline) or not np.isfinite(sensor_baseline) else deck_baseline - sensor_baseline
    aligned_y = item['raw_y'] + shift
    aligned_trend = item['raw_trend'] + shift
    for event_sequence, value in zip(item['event_sequence'], aligned_y):
        aligned_rows.append({
            'deck': deck,
            'sensor_alias': sensor_alias,
            'event_sequence': int(event_sequence),
            'aligned_sensor_health_score': float(value) if np.isfinite(value) else np.nan,
            'baseline_shift': float(shift),
        })

overall_aligned_sensor_scores = pd.DataFrame(aligned_rows)
overall_str_event_trend = (
    overall_aligned_sensor_scores
    .groupby(['deck', 'event_sequence'], as_index=False)
    .agg(
        overall_mean_health=('aligned_sensor_health_score', 'mean'),
        overall_median_health=('aligned_sensor_health_score', 'median'),
        overall_sensor_sd=('aligned_sensor_health_score', 'std'),
        sensor_count=('sensor_alias', 'nunique'),
    )
)

fig, ax = plt.subplots(1, 1, figsize=(9.6, 5.8), constrained_layout=False)
fig.subplots_adjust(left=0.085, right=0.985, bottom=0.250, top=0.820)
fig.suptitle('3.6 Overall ACC-Normalized STR Health Score Trend', fontsize=PLOT_TITLE_FONTSIZE, y=0.940)

summary_rows = []
for deck in ['NEW', 'OLD']:
    d = overall_str_event_trend[overall_str_event_trend['deck'].eq(deck)].sort_values('event_sequence').copy()
    color = DECK_COLORS[deck]
    x = d['event_sequence'].to_numpy(float)
    y = d['overall_mean_health'].to_numpy(float)
    finite = np.isfinite(y)
    trend = _low_frequency_health_trend(y)

    ax.scatter(x[finite], y[finite], s=4.0, color=color, alpha=0.075, linewidths=0, rasterized=True)
    ax.plot(x, trend, color=color, lw=2.7, label=deck)

    summary_rows.append({
        'deck': deck,
        'event_count': int(len(d)),
        'sensor_count_per_event': int(d['sensor_count'].median()) if len(d) else 0,
        'aligned_mean_health_median': float(np.nanmedian(y)) if finite.any() else np.nan,
        'aligned_mean_health_sd': float(np.nanstd(y, ddof=1)) if finite.sum() > 1 else np.nan,
        'trend_start': float(trend[np.isfinite(trend)][0]) if np.isfinite(trend).any() else np.nan,
        'trend_end': float(trend[np.isfinite(trend)][-1]) if np.isfinite(trend).any() else np.nan,
        'trend_end_minus_start': float(trend[np.isfinite(trend)][-1] - trend[np.isfinite(trend)][0]) if np.isfinite(trend).any() else np.nan,
    })

ax.set_title('G1 STR DO + G2 STR UP\nmedian-level aligned mean of 12 STR sensors', fontsize=PANEL_TITLE_FONTSIZE, pad=8)
ax.set_ylim(55, 80)
ax.set_xlabel('Event sequence', fontsize=10)
ax.set_ylabel('Displayed STR health score', fontsize=10)
ax.grid(True, alpha=0.24, linewidth=0.6)
ax.tick_params(axis='both', labelsize=9.0, length=2.5, pad=1.5)

legend_handles = [
    Line2D([0], [0], color=DECK_COLORS['NEW'], lw=2.8, label='NEW'),
    Line2D([0], [0], color=DECK_COLORS['OLD'], lw=2.8, label='OLD'),
]
fig.text(
    0.50,
    0.145,
    'Sensor median levels are aligned before averaging; the score calculation itself is unchanged.',
    ha='center',
    fontsize=10.5,
)
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.055),
    ncol=2,
    frameon=True,
    fontsize=11,
)
plt.show()

overall_str_trend_summary = pd.DataFrame(summary_rows)
print('Overall STR trend summary after sensor-level median alignment')
display(overall_str_trend_summary.round(3))

## Interpretation

The key point of this notebook is that the STR response is not compared directly in its raw spectral-response form. Instead, it is first divided by the ACC class input measured at the same event, span, and side. This removes part of the class-wise excitation variability before sensor similarity is calculated, making the similarity score more representative of the strain response relative to the measured input condition.

For the NEW deck, the event-score standard deviation decreases in both G1 and G2 after ACC input normalization. This indicates that STR-only comparison leaves larger within-group or event-to-event score variability, whereas ACC-assisted input correction stabilizes the G1/G2 score distributions.

The method is defined by the physical normalization rule `r_hat_STR = r_tilde_STR / A_ACC`, without manual thresholds, outlier filters, or arbitrary score ranges. Therefore, ACC channels are used as excitation-normalization references, while the final structural response assessment is performed on the STR G1/G2 sensor groups.
